# Label geometry: generalising the ramp win

The soft ramp (+0.0074 cloud, confirmed) works because the hard label `y=1 at t>=tau` is noisy
right after a break — the signal has not emerged yet. But the ramp we shipped is **linear with a
fixed width** (`W=40`), which is arbitrary. Two ways the true detectability curve should differ:

**1. Shape.** Detection evidence accumulates like a random walk — the standardized signal grows
~`sqrt(n)` in the number of post-break points. So a CONCAVE target (rises fast, saturates) may
match reality better than a straight line. Convex and sigmoid are tested as controls.

**2. Width should depend on BREAK MAGNITUDE.** A large variance jump is visible within a few
points; a subtle one takes hundreds. Classical detection theory: samples-to-detect scales like
`1/effect^2`. A single fixed `W=40` forces the same detectability lag on every series. Here
`W_i` is scaled per series by the measured break magnitude (centred so the median series keeps
W=40, i.e. the confirmed baseline).

**Not leakage:** the magnitude uses `tau` and the series' own values, but it only shapes the
TRAINING target — exactly like `tau` already does for the ramp. Validation rows are always
scored against the hard label `y`, and the model sees no magnitude at inference.

Believe a win only at **t >= 3**, then re-run at 10 folds (the rule that saved us twice).

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
from catboost import CatBoostRegressor

TOOLS = os.path.abspath('../tools'); sys.path.insert(0, TOOLS)
import cv_tools as CV
REPO = os.path.abspath('../../..')

cache = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = cache['X'], cache['y'], cache['sid'], cache['time']
sampled = cache['is_sampled_step']
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
step = CV.steps_from_index(index)
uids = np.unique(sid)

idx = pd.read_parquet(os.path.join(REPO, 'y_train_index.parquet'))
tau_of = idx['tau_index'].reindex(uids)
tau_row = tau_of.reindex(sid).to_numpy()
since = step - tau_row
is_break = tau_row >= 0
print(f'{len(y):,} rows | {int(is_break[np.unique(sid, return_index=True)[1]].sum()):,} break series')

5,036,517 rows | 4,967 break series


In [2]:
# ---- per-series break MAGNITUDE (post-break segment vs the reference) ----
MAG_CACHE = 'break_magnitude.npz'
if os.path.exists(MAG_CACHE):
    mg = np.load(MAG_CACHE); mag_of = pd.Series(mg['mag'], index=mg['uid'])
    print('loaded cached magnitudes')
else:
    Xr = pd.read_parquet(os.path.join(REPO, 'X_train.parquet'))
    ids = Xr.index.get_level_values('id').to_numpy(); per = Xr['period'].to_numpy()
    val = Xr['value'].to_numpy(np.float64)
    us, starts = np.unique(ids, return_index=True); bounds = np.append(starts, len(ids))
    mags = {}
    for k in range(len(us)):
        s, e = bounds[k], bounds[k + 1]; m = per[s:e] == 2
        ref = val[s:e][~m]; on = val[s:e][m]
        ti = int(tau_of.get(us[k], -1)) if us[k] in tau_of.index else -1
        if ti < 0 or len(on) - ti < 5 or len(ref) < 8:
            mags[us[k]] = np.nan; continue
        post = on[ti:]
        mu, sd = ref.mean(), ref.std() + 1e-9
        # effect size of the break vs the reference: variance ratio + standardized mean shift
        vr = abs(np.log((post.var() + 1e-9) / (ref.var() + 1e-9)))
        ms = abs(post.mean() - mu) / sd
        mags[us[k]] = float(np.hypot(vr, ms))
    mag_of = pd.Series(mags)
    np.savez(MAG_CACHE, uid=mag_of.index.to_numpy(), mag=mag_of.to_numpy())
    print('built magnitudes')

m = mag_of.dropna()
print(f'magnitude: median {m.median():.3f} | q10 {m.quantile(.1):.3f} | q90 {m.quantile(.9):.3f}')

# samples-to-detect ~ 1/effect^2, centred so the MEDIAN series keeps W=40 (the confirmed baseline)
W0, MED = 40.0, float(m.median())
w_series = (W0 * (MED ** 2) / np.maximum(mag_of.reindex(uids).to_numpy(), 1e-3) ** 2)
w_series = np.clip(w_series, 5.0, 200.0)
w_row = pd.Series(w_series, index=uids).reindex(sid).to_numpy()
print(f'per-series W: median {np.nanmedian(w_series):.0f} | q10 {np.nanquantile(w_series,.1):.0f} '
      f'| q90 {np.nanquantile(w_series,.9):.0f}')

loaded cached magnitudes
magnitude: median 0.243 | q10 0.064 | q90 0.818
per-series W: median 40 | q10 5 | q90 200


In [3]:
def make_target(kind, W=40.0, w_vec=None):
    """Soft target in [0,1]: 0 before the break, rising to 1 as the signal becomes detectable."""
    w = (np.full(len(since), W, dtype=np.float64) if w_vec is None
         else np.nan_to_num(w_vec, nan=W))
    u = np.clip(since / (2.0 * w), 0.0, 1.0)          # progress through the ramp
    if kind == 'linear':
        t = u
    elif kind == 'concave':                            # sqrt(n) evidence growth
        t = np.sqrt(u)
    elif kind == 'convex':
        t = u * u
    elif kind == 'sigmoid':
        t = 1.0 / (1.0 + np.exp(-(since - w) / np.maximum(w / 3.0, 1e-6)))
        t = np.clip(t, 0.0, 1.0)
    else:
        raise ValueError(kind)
    t = np.asarray(t, dtype=np.float64)
    t[~is_break] = 0.0
    t[is_break & (since < 0)] = 0.0                   # nothing has happened yet
    return pd.Series(t, index=index)

TARGETS = {
    'linear_W40':   make_target('linear'),            # the confirmed baseline
    'concave_W40':  make_target('concave'),
    'convex_W40':   make_target('convex'),
    'sigmoid_W40':  make_target('sigmoid'),
    'linear_magW':  make_target('linear', w_vec=w_row),
    'concave_magW': make_target('concave', w_vec=w_row),
}
for n, t in TARGETS.items():
    tv = t.to_numpy()[sampled]
    print(f'{n:14s} mean {tv.mean():.3f} | in-ramp rows {(tv>0).sum()-(tv>=1).sum():,}')

linear_W40     mean 0.072 | in-ramp rows 19,071
concave_W40    mean 0.082 | in-ramp rows 19,071
convex_W40     mean 0.063 | in-ramp rows 19,071
sigmoid_W40    mean 0.071 | in-ramp rows 35,223
linear_magW    mean 0.066 | in-ramp rows 21,062
concave_magW   mean 0.078 | in-ramp rows 21,062


In [4]:
REG = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
           bootstrap_type='Bernoulli', subsample=0.8, loss_function='RMSE',
           random_seed=0, verbose=0, thread_count=-1)

def fp(tgt):
    def _f(train, val):
        m = CatBoostRegressor(**REG)
        m.fit(train.X, tgt.loc[train.index].to_numpy(), sample_weight=train.w)
        return m.predict(val.X)
    return _f

folds = CV.series_folds(sid, n_splits=5, seed=0)
OUT = 'geometry_results.json'
done = json.load(open(OUT)) if os.path.exists(OUT) else {}
res = {}
for name, tgt in TARGETS.items():
    t0 = time.time(); print(f'\n=== {name} ===', flush=True)
    r = CV.run_cv(X, y, sid, index, fp(tgt), folds=folds,
                  train_row_mask=sampled, row_step=step, verbose=False)
    res[name] = r
    done[name] = r.fold_scores.tolist(); json.dump(done, open(OUT, 'w'), indent=2)
    print(f'{r.summary(name)} | {time.time()-t0:.0f}s', flush=True)


=== linear_W40 ===


linear_W40         mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297 | 165s



=== concave_W40 ===


concave_W40        mean 0.6078 +/- 0.0154 (sem 0.0069) | OOF 0.6069 | folds: 0.5959  0.6027  0.5933  0.6171  0.6299 | 171s



=== convex_W40 ===


...

sigmoid_W40        mean 0.6082 +/- 0.0161 (sem 0.0072) | OOF 0.6072 | folds: 0.5934  0.6023  0.5947  0.6212  0.6292 | 181s



=== linear_magW ===


linear_magW        mean 0.6029 +/- 0.0115 (sem 0.0051) | OOF 0.6024 | folds: 0.5994  0.5864  0.6045  0.6065  0.6180 | 168s



=== concave_magW ===


concave_magW       mean 0.6063 +/- 0.0112 (sem 0.0050) | OOF 0.6056 | folds: 0.6025  0.5931  0.6022  0.6104  0.6232 | 166s


In [5]:
print(CV.summarize(res))
print('\n--- each geometry vs the shipped linear_W40 ---')
for n in res:
    if n != 'linear_W40':
        print(CV.paired_compare(res[n], res['linear_W40'], n, 'linear_W40'))

linear_W40         mean 0.6089 +/- 0.0157 (sem 0.0070) | OOF 0.6079 | folds: 0.5940  0.6044  0.5958  0.6206  0.6297
sigmoid_W40        mean 0.6082 +/- 0.0161 (sem 0.0072) | OOF 0.6072 | folds: 0.5934  0.6023  0.5947  0.6212  0.6292
concave_W40        mean 0.6078 +/- 0.0154 (sem 0.0069) | OOF 0.6069 | folds: 0.5959  0.6027  0.5933  0.6171  0.6299
convex_W40         mean 0.6075 +/- 0.0159 (sem 0.0071) | OOF 0.6066 | folds: 0.5923  0.6023  0.5948  0.6186  0.6293
concave_magW       mean 0.6063 +/- 0.0112 (sem 0.0050) | OOF 0.6056 | folds: 0.6025  0.5931  0.6022  0.6104  0.6232
linear_magW        mean 0.6029 +/- 0.0115 (sem 0.0051) | OOF 0.6024 | folds: 0.5994  0.5864  0.6045  0.6065  0.6180

--- each geometry vs the shipped linear_W40 ---
concave_W40 - linear_W40: -0.0011 +/- 0.0022 (sem 0.0010, t -1.16, wins 2/5) -> within noise
  per-fold deltas: +0.0019  -0.0017  -0.0025  -0.0036  +0.0002
convex_W40 - linear_W40: -0.0014 +/- 0.0007 (sem 0.0003, t -4.45, wins 0/5) -> consistent
  per-fol